In [1]:
import torch
import time

# 读取文本
input_file = "datasets/to-live-a-novel-cleaned.txt"
with open(input_file, "r", encoding="utf-8") as f:
    text = f.read()
print(f"text length: {len(text)}")

# 构建词汇表，建立字符与索引之间的双向映射
chars_list = sorted(list(set(text)))
char2idx = {c: i for i, c in enumerate(chars_list)}
idx2char = {i: c for i, c in enumerate(chars_list)}
vocab_size = len(chars_list)
print(f"词汇表大小: {vocab_size}")


class CharDataset(torch.utils.data.Dataset):
    """数据集类，返回整数索引张量，one-hot 编码推迟到 GPU 上执行"""

    def __init__(self, text, char2idx, learn_char_len=128, step_char_len=1):
        self.char2idx = char2idx
        self.vocab_size = len(char2idx)
        self.learn_char_len = learn_char_len
        # 将整篇文本转为索引列表
        self.data = [char2idx[c] for c in text if c in char2idx]
        self.samples = []
        # 用滑动窗口切出训练样本
        for i in range(0, len(self.data) - learn_char_len, step_char_len):
            x_idx = self.data[i : i + learn_char_len]
            y_idx = self.data[i + 1 : i + learn_char_len + 1]
            self.samples.append((x_idx, y_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x_idx, y_idx = self.samples[idx]
        # 返回整数索引，不在此处做 one-hot
        x_tensor = torch.tensor(x_idx, dtype=torch.long)
        y_tensor = torch.tensor(y_idx, dtype=torch.long)
        return x_tensor, y_tensor


class CharRNN(torch.nn.Module):
    """使用 torch.nn.RNN 构建的字符级循环神经网络"""

    def __init__(self, vocab_size, hidden_size, num_layers=1):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        # torch.nn.RNN 内部已封装 W_xh、W_hh、偏置和 tanh 激活
        self.rnn = torch.nn.RNN(
            input_size=vocab_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            nonlinearity='tanh'
        )
        # 输出层：将隐状态投影回词汇表维度
        self.fc = torch.nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h_prev=None):
        # x 形状：(batch, seq_len, vocab_size)，已是 one-hot
        batch_size = x.size(0)
        if h_prev is None:
            h_prev = torch.zeros(self.num_layers, batch_size, self.hidden_size, device=x.device)
        # torch.nn.RNN 一次性处理整个序列，无需手动循环
        out, h = self.rnn(x, h_prev)  # out: (batch, seq_len, hidden_size)
        outputs = self.fc(out)         # (batch, seq_len, vocab_size)
        return outputs, h


# 选择设备
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# 初始化模型
hidden_size = 1024
model = CharRNN(vocab_size=vocab_size, hidden_size=hidden_size).to(device)

# 构建数据集和数据加载器
char_dataset = CharDataset(text, char2idx=char2idx, learn_char_len=128)
dataloader = torch.utils.data.DataLoader(
    char_dataset, batch_size=64, num_workers=8, shuffle=True, pin_memory=True
)

# 初始化优化器和损失函数
learning_rate = 0.001
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
loss_fn = torch.nn.CrossEntropyLoss()

total_batches = len(dataloader)
num_epochs = 20

# 训练循环
for epoch in range(num_epochs):
    total_loss = 0
    batch_count = 0
    for batch_idx, (x_idx_batch, y_idx_batch) in enumerate(dataloader):
        # 整数索引搬到 GPU
        x_idx_batch = x_idx_batch.to(device, non_blocking=True)
        y_idx_batch = y_idx_batch.to(device, non_blocking=True)

        # 在 GPU 上执行 one-hot 编码
        x_one_hot = torch.nn.functional.one_hot(x_idx_batch, num_classes=vocab_size).float()
        # y 直接用整数索引作为 CrossEntropyLoss 的目标，无需 one-hot

        # 前向传播
        outputs, _ = model(x_one_hot)

        # 将预测和目标拍平以适配 CrossEntropyLoss
        preds = outputs.reshape(-1, vocab_size)
        targets = y_idx_batch.reshape(-1)
        loss = loss_fn(preds, targets)

        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        # 梯度裁剪
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item()
        batch_count += 1

        remaining_batches = total_batches - (batch_idx + 1)
        eta = remaining_batches * 0.09
        print(f"\rEpoch [{epoch+1}/{num_epochs}] | Batch [{batch_idx+1}/{total_batches}] | 损失: {loss.item():.4f} | 预计剩余: {eta:.1f}s", end="")

    print(f"\rEpoch [{epoch+1}/{num_epochs}] 完成 | 平均损失: {total_loss / batch_count:.4f}")


def generate_text(model, char2idx, idx2char, seed_text, gen_len=80):
    """给定种子文本，用模型逐字符生成续写"""
    model.eval()
    vocab_size = len(char2idx)
    device = next(model.parameters()).device

    seed_indices = [char2idx[c] for c in seed_text if c in char2idx]

    # 用种子文本初始化隐状态
    h = None
    for idx in seed_indices:
        # 在 GPU 上执行 one-hot 编码
        x_t = torch.nn.functional.one_hot(
            torch.tensor(idx, device=device), num_classes=vocab_size
        ).float().unsqueeze(0).unsqueeze(0)
        _, h = model(x_t, h)

    generated = list(seed_text)
    last_idx = seed_indices[-1] if seed_indices else 0

    # 逐字符生成
    for _ in range(gen_len):
        # 在 GPU 上执行 one-hot 编码
        x_t = torch.nn.functional.one_hot(
            torch.tensor(last_idx, device=device), num_classes=vocab_size
        ).float().unsqueeze(0).unsqueeze(0)
        y_t, h = model(x_t, h)
        next_idx = y_t.squeeze(0).squeeze(0).argmax().item()
        generated.append(idx2char[next_idx])
        last_idx = next_idx

    return "".join(generated)


# 测试生成
seed = "凤霞命苦啊，"
text_gen = generate_text(model, char2idx, idx2char, seed_text=seed, gen_len=80)
print(f"\n种子: '{seed}'")
print(f"生成: '{text_gen}'")
print("-" * 50)


text length: 98634
词汇表大小: 1863
Using device: cuda
Epoch [1/20] 完成 | 平均损失: 3.14950] | 损失: 1.6163 | 预计剩余: 0.0ss
Epoch [2/20] 完成 | 平均损失: 1.10270] | 损失: 0.7163 | 预计剩余: 0.0ss
Epoch [3/20] 完成 | 平均损失: 0.47910] | 损失: 0.3787 | 预计剩余: 0.0ss
Epoch [4/20] 完成 | 平均损失: 0.28170] | 损失: 0.2198 | 预计剩余: 0.0ss
Epoch [5/20] 完成 | 平均损失: 0.22310] | 损失: 0.2166 | 预计剩余: 0.0ss
Epoch [6/20] 完成 | 平均损失: 0.19310] | 损失: 0.1990 | 预计剩余: 0.0ss
Epoch [7/20] 完成 | 平均损失: 0.17660] | 损失: 0.1581 | 预计剩余: 0.0ss
Epoch [8/20] 完成 | 平均损失: 0.16340] | 损失: 0.1724 | 预计剩余: 0.0ss
Epoch [9/20] 完成 | 平均损失: 0.15770] | 损失: 0.1217 | 预计剩余: 0.0ss
Epoch [10/20] 完成 | 平均损失: 0.14950] | 损失: 0.1221 | 预计剩余: 0.0ss
Epoch [11/20] 完成 | 平均损失: 0.13920] | 损失: 0.1467 | 预计剩余: 0.0ss
Epoch [12/20] 完成 | 平均损失: 0.13840] | 损失: 0.1266 | 预计剩余: 0.0ss
Epoch [13/20] 完成 | 平均损失: 0.13440] | 损失: 0.1153 | 预计剩余: 0.0ss
Epoch [14/20] 完成 | 平均损失: 0.13330] | 损失: 0.1420 | 预计剩余: 0.0ss
Epoch [15/20] 完成 | 平均损失: 0.12410] | 损失: 0.1278 | 预计剩余: 0.0ss
Epoch [16/20] 完成 | 平均损失: 0.12520] | 损失: 0.10

In [2]:

# 测试生成
seed = "凤霞命苦啊，"
text_gen = generate_text(model, char2idx, idx2char, seed_text=seed, gen_len=80)
print(f"\n种子: '{seed}'")
print(f"生成: '{text_gen}'")
print("-" * 50)



种子: '凤霞命苦啊，'
生成: '凤霞命苦啊，你把鞋弄破了，想着有庆也没有。<EOS>这村里谁都没看到了现在里面的声音。<EOS>"这时我女儿凤霞推门进来，又摇摇晃晃地把门关上。<EOS>凤霞尖声细气地对我'
--------------------------------------------------
